####Data Reading


In [0]:
df=spark.sql("select * from databricks_catalog.silver.orders_silver")
df.display()

#####Dimension Tables

In [0]:
%sql
select * from databricks_catalog.gold.dimproducts

In [0]:
df_dimcustomers=spark.sql("select DimCustomerKey,customer_id as dim_customer_id from databricks_catalog.gold.dimcustomers")
df_dimproducts=spark.sql("select product_id as DimProductKey,product_id as dim_product_id from databricks_catalog.gold.dimproducts")

####Fact dataframe

In [0]:
df_fact=df.join(df_dimcustomers,df["customer_id"]==df_dimcustomers["dim_customer_id"],"left").join(df_dimproducts,df["product_id"]==df_dimproducts["dim_product_id"],"left")
df_fact_new=df_fact.drop("dim_customer_id","dim_product_id","customer_id","product_id")


####Upsert

In [0]:
from delta.tables import DeltaTable
if spark.catalog.tableExists("databricks_catalog.gold.FactOrders"):
    dlt_obj=DeltaTable.forName(spark,"databricks_catalog.gold.FactOrders")
    dlt_obj.alias("trg").merge(df_fact_new.alias("src"),"trg.order_id= src.order_id and trg.DimCustomerKey=src.DimCustomerKey and trg.DimProductKey=src.DimProductKey")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
            .execute()
else:
    df_fact_new.write.format('delta')\
        .option("path","abfss://gold@datalakedatabricksram.df.core.windows.net/FactOrders")\
        .saveAsTable("databricks_catalog.gold.FactOrders")